In [21]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

FILE_NAME = "p0.8_mu0.2_1"
emb_path = f"../pretrain/{FILE_NAME}.npy"
#emb_path = '/Users/acw721/Desktop/research/linkstream/pretrain/node_level_feat/node_feat.npy'
interaction_path = f"../syn_data/{FILE_NAME}.csv"
K = 5

X = np.load(emb_path).astype(np.float32)   # shape: [num_nodes, dim]


node_ids = np.arange(X.shape[0])

kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

node2label = pd.DataFrame({
    "node": node_ids,
    "label": labels
})

inter_df = pd.read_csv(interaction_path)
inter_df = inter_df[["source", "destination", "timestamp"]].copy()

inter_df["source"] = inter_df["source"].astype(int)
inter_df["destination"] = inter_df["destination"].astype(int)

src_label_df = node2label.rename(columns={
    "node": "source",
    "label": "source_commu"
})
result_df = inter_df.merge(src_label_df, on="source", how="left")

dst_label_df = node2label.rename(columns={
    "node": "destination",
    "label": "destination_commu"
})
result_df = result_df.merge(dst_label_df, on="destination", how="left")

result_df = result_df[
    ["source", "destination", "timestamp", "source_commu", "destination_commu"]
].copy()

out_dir = os.path.join("results", FILE_NAME)
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ctdne.csv")
result_df.to_csv(out_path, index=False)

print(result_df.head(20))
print(f"saved to {out_path}")
print("rows =", len(result_df))
print("missing source_commu =", result_df["source_commu"].isna().sum())
print("missing destination_commu =", result_df["destination_commu"].isna().sum())

    source  destination  timestamp  source_commu  destination_commu
0       16           19          5             1                  1
1        6           46         11             3                  1
2       16           28         18             1                  1
3        5           22         25             3                  1
4        8           35         31             4                  1
5       12           28         37             1                  1
6       15           41         43             1                  1
7        7           49         52             1                  3
8       38           43         60             0                  0
9       24           45         63             2                  1
10       0           45         70             1                  1
11       6           47         77             3                  1
12       6           40         83             3                  2
13      15           21         87             1